## 1.0 Importação das Bibliotecas

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import keras


from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from category_encoders import BinaryEncoder

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 2.0 Carregamento dos Dados

In [8]:
ch = pd.read_csv('/kaggle/input/playground-series-s6e3/train.csv', index_col='id')
print(ch.shape)
ch.head()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/playground-series-s6e3/train.csv'

### 2.1 Distribuição do Target

Desbalanceamento: ~77.5% No / ~22.5% Yes — estratificação obrigatória.

In [ ]:
ch['Churn'].value_counts(normalize=True)

### 2.2 Divisão Treino/Teste

Dado o desbalanceamento da variável target, é necessário o uso de `stratify`.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    ch.drop('Churn', axis=1),
    ch['Churn'].map({'No': 0, 'Yes': 1}),
    test_size=0.2,
    random_state=42,
    stratify=ch['Churn']
)
x_train.shape, x_test.shape

## 3.0 Análise Descritiva

In [ ]:
x_train.describe(include='object').T.sort_values('unique')

In [ ]:
x_train.describe(exclude='object').T.sort_values('mean')

In [ ]:
cat_vars = x_train.describe(include='object').columns
num_vars = x_train.describe(exclude='object').columns

In [ ]:
x_train[num_vars].corr()

In [ ]:
fig, axes = plt.subplots(nrows=5, ncols=3, figsize=(18, 20))
axes = axes.flatten()

for i, col in enumerate(cat_vars):
    x_train[col].value_counts(normalize=True).plot(kind='bar', ax=axes[i], title=col)
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].set_xlabel('')

plt.tight_layout(pad=3.0)
plt.show()

In [ ]:
x_train[num_vars].hist(figsize=(15, 10))
plt.tight_layout()
plt.show()

In [ ]:
# SeniorCitizen é binário (0/1) — tratar como categórica
num_vars = num_vars.drop('SeniorCitizen')
cat_vars = pd.Index(cat_vars.tolist() + ['SeniorCitizen'])

## 4.0 Pré-processamento

Aplicamos aqui um baseline de pré-processamento simples: **OneHotEncoder** para variáveis categóricas e **StandardScaler** para variáveis numéricas.

O objetivo desta etapa não é otimizar o preprocessamento, mas sim ter uma base justa para comparar os modelos. A partir dos **top 3 modelos** identificados na Fase 1, avançaremos para testes com diferentes encoders, estratégias de feature engineering e, por fim, fine-tuning com Optuna.

In [ ]:
cat_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

num_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_vars),
    ('cat', cat_pipeline, cat_vars)
])

## 5.0 Fase 1 — Baseline: Ranqueamento dos Modelos

Preprocessamento fixo (OneHotEncoder + StandardScaler) aplicado a todos os modelos.
Objetivo: identificar os **top 3** para tuning nas fases seguintes.

In [ ]:
modelos = {
    'LogReg':   LogisticRegression(max_iter=1000, random_state=42),
    'RF':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGB':      XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0),
    'LGBM':     LGBMClassifier(random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados_fase1 = []

for model_name, modelo in modelos.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', modelo)
    ])
    scores = cross_val_score(pipe, x_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    resultados_fase1.append({
        'Model':        model_name,
        'ROC-AUC Mean': scores.mean(),
        'ROC-AUC Std':  scores.std(),
    })
    print(f'{model_name:<12}  →  {scores.mean():.4f} ± {scores.std():.4f}')

df_fase1 = (
    pd.DataFrame(resultados_fase1)
    .sort_values('ROC-AUC Mean', ascending=False)
    .reset_index(drop=True)
)
print('\n--- Ranking Final ---')
df_fase1

## 6.0 Fase 2 — Feature Engineering + Encoders

Usando os **top 3 modelos** da Fase 1 (CatBoost, XGB, LGBM), testamos todas as combinações de:
- **Feature sets:** original vs. com features criadas manualmente
- **Encoders:** OneHotEncoder vs. BinaryEncoder (sem scaler — árvores são invariantes à escala)

Total: 2 × 2 × 3 = **12 combinações**

In [ ]:
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

def create_features(df):
    df = df.copy()
    df['charges_per_tenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)
    df['charges_diff']       = df['TotalCharges'] - (df['MonthlyCharges'] * df['tenure'])
    df['is_new_customer']    = (df['tenure'] <= 12).astype(int)
    df['is_monthly']         = (df['Contract'] == 'Month-to-month').astype(int)
    df['n_services']         = (df[service_cols] == 'Yes').sum(axis=1)
    df['senior_no_support']  = ((df['SeniorCitizen'] == 1) & (df['TechSupport'] == 'No')).astype(int)
    return df

new_num_features = ['charges_per_tenure', 'charges_diff', 'is_new_customer',
                    'is_monthly', 'n_services', 'senior_no_support']

x_train_feat = create_features(x_train)
num_vars_feat = num_vars.tolist() + new_num_features
cat_vars_feat = cat_vars.tolist()

print(f'Features originais:   {len(num_vars) + len(cat_vars)}')
print(f'Features engineered:  {len(num_vars_feat) + len(cat_vars_feat)}')

In [ ]:
def make_prep(n_vars, c_vars, encoder):
    prep = ColumnTransformer(transformers=[
        ('num', 'passthrough', n_vars),
        ('cat', encoder, c_vars),
    ], verbose_feature_names_out=False)
    prep.set_output(transform='pandas')
    return prep

encoders_grid = {
    'OneHot': OneHotEncoder(handle_unknown='ignore', sparse_output=False),
    'Binary': BinaryEncoder(),
}

feature_sets = {
    'Original':   (x_train,      num_vars.tolist(),  cat_vars.tolist()),
    'Engineered': (x_train_feat, num_vars_feat,       cat_vars_feat),
}

top3 = {
    'XGB':      XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0),
    'LGBM':     LGBMClassifier(random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados_fase2 = []

for feat_name, (X, n_vars, c_vars) in feature_sets.items():
    for enc_name, encoder in encoders_grid.items():
        for model_name, modelo in top3.items():
            pipe = Pipeline([
                ('prep',  make_prep(n_vars, c_vars, clone(encoder))),
                ('model', clone(modelo))
            ])
            scores = cross_val_score(pipe, X, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
            resultados_fase2.append({
                'Features':     feat_name,
                'Encoder':      enc_name,
                'Model':        model_name,
                'ROC-AUC Mean': scores.mean(),
                'ROC-AUC Std':  scores.std(),
            })
            print(f'{feat_name:<12} | {enc_name:<7} | {model_name:<9}  →  {scores.mean():.4f} ± {scores.std():.4f}')

df_fase2 = (
    pd.DataFrame(resultados_fase2)
    .sort_values('ROC-AUC Mean', ascending=False)
    .reset_index(drop=True)
)
print('\n--- Top 10 Combinações ---')
df_fase2.head(10)

In [ ]:
best_per_model = (
    df_fase2
    .sort_values('ROC-AUC Mean', ascending=False)
    .groupby('Model')
    .first()
    .reset_index()
)
print(best_per_model[['Model', 'Features', 'Encoder', 'ROC-AUC Mean']].to_string(index=False))

## 7.0 Fase 3 — Fine-tuning com Optuna

Tuning dos top 3 modelos usando a melhor configuração encontrada na Fase 2.
Foco nos hiperparâmetros de maior impacto em GBMs.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

configs = {
    'CatBoost': {'X': x_train,      'n_vars': num_vars.tolist(), 'c_vars': cat_vars.tolist()},
    'XGB':      {'X': x_train,      'n_vars': num_vars.tolist(), 'c_vars': cat_vars.tolist()},
    'LGBM':     {'X': x_train_feat, 'n_vars': num_vars_feat,     'c_vars': cat_vars_feat},
}

def make_prep_onehot(n_vars, c_vars):
    prep = ColumnTransformer(transformers=[
        ('num', 'passthrough', n_vars),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), c_vars),
    ], verbose_feature_names_out=False)
    prep.set_output(transform='pandas')
    return prep

def objective_xgb(trial, X, n_vars, c_vars):
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':     trial.suggest_int('max_depth', 3, 8),
        'subsample':     trial.suggest_float('subsample', 0.6, 1.0),
    }
    pipe = Pipeline([('prep', make_prep_onehot(n_vars, c_vars)),
                     ('model', XGBClassifier(**params, eval_metric='logloss', random_state=42, verbosity=0))])
    return cross_val_score(pipe, X, y_train, cv=cv, scoring='roc_auc', n_jobs=-1).mean()

def objective_lgbm(trial, X, n_vars, c_vars):
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':     trial.suggest_int('max_depth', 3, 8),
        'subsample':     trial.suggest_float('subsample', 0.6, 1.0),
    }
    pipe = Pipeline([('prep', make_prep_onehot(n_vars, c_vars)),
                     ('model', LGBMClassifier(**params, random_state=42, verbose=-1))])
    return cross_val_score(pipe, X, y_train, cv=cv, scoring='roc_auc', n_jobs=-1).mean()

def objective_catboost(trial, X, n_vars, c_vars):
    params = {
        'iterations':    trial.suggest_int('iterations', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth':         trial.suggest_int('depth', 3, 8),
        'subsample':     trial.suggest_float('subsample', 0.6, 1.0),
    }
    pipe = Pipeline([('prep', make_prep_onehot(n_vars, c_vars)),
                     ('model', CatBoostClassifier(**params, random_state=42, verbose=0))])
    return cross_val_score(pipe, X, y_train, cv=cv, scoring='roc_auc', n_jobs=-1).mean()

objectives = {
    'XGB':      objective_xgb,
    'LGBM':     objective_lgbm,
    'CatBoost': objective_catboost,
}

N_TRIALS = 100
melhores_params = {}

for model_name, cfg in configs.items():
    print(f'Otimizando {model_name}...')
    study = optuna.create_study(direction='maximize')
    study.optimize(
        lambda trial, m=model_name, c=cfg: objectives[m](trial, c['X'], c['n_vars'], c['c_vars']),
        n_trials=N_TRIALS,
        show_progress_bar=True,
    )
    melhores_params[model_name] = study.best_params
    print(f'  → Melhor AUC: {study.best_value:.4f} | Params: {study.best_params}\n')

## 8.0 Submissão

Treina os modelos tunados no dataset completo e gera:
- `submission_individual.csv` — melhor modelo individual (CatBoost)
- `submission_ensemble.csv` — média das probabilidades dos 3 modelos

In [ ]:
X_full      = ch.drop('Churn', axis=1)
y_full      = ch['Churn'].map({'No': 0, 'Yes': 1})
X_full_feat = create_features(X_full)

test      = pd.read_csv('/kaggle/input/playground-series-s6e3/test.csv', index_col='id')
test_feat = create_features(test)

modelos_tunados = {
    'CatBoost': CatBoostClassifier(**melhores_params['CatBoost'], random_state=42, verbose=0),
    'XGB':      XGBClassifier(**melhores_params['XGB'], eval_metric='logloss', random_state=42, verbosity=0),
    'LGBM':     LGBMClassifier(**melhores_params['LGBM'], random_state=42, verbose=-1),
}

configs_full = {
    'CatBoost': {'X_train': X_full,      'X_test': test,      'n_vars': num_vars.tolist(), 'c_vars': cat_vars.tolist()},
    'XGB':      {'X_train': X_full,      'X_test': test,      'n_vars': num_vars.tolist(), 'c_vars': cat_vars.tolist()},
    'LGBM':     {'X_train': X_full_feat, 'X_test': test_feat, 'n_vars': num_vars_feat,     'c_vars': cat_vars_feat},
}

preds = {}
for model_name, modelo in modelos_tunados.items():
    cfg  = configs_full[model_name]
    pipe = Pipeline([
        ('prep',  make_prep_onehot(cfg['n_vars'], cfg['c_vars'])),
        ('model', modelo)
    ])
    pipe.fit(cfg['X_train'], y_full)
    preds[model_name] = pipe.predict_proba(cfg['X_test'])[:, 1]
    print(f'{model_name} treinado.')

# Submissão individual
sub_individual = pd.DataFrame({'id': test.index, 'Churn': preds['CatBoost']})
sub_individual.to_csv('submission_individual.csv', index=False)

# Submissão ensemble
pred_ensemble = (preds['CatBoost'] + preds['XGB'] + preds['LGBM']) / 3
sub_ensemble  = pd.DataFrame({'id': test.index, 'Churn': pred_ensemble})
sub_ensemble.to_csv('submission_ensemble.csv', index=False)

print('\nArquivos gerados: submission_individual.csv | submission_ensemble.csv')
sub_ensemble.head()

### 9.0 Aplicação de redes neurais! Usando o Keras.



In [ ]:
X_train,

## 9.0 Conclusões e Possíveis Melhorias

### Resultados

| Fase | Melhor AUC | Modelo |
|---|---|---|
| Baseline (Fase 1) | 0.9158 | CatBoost |
| Feature Eng. + Encoders (Fase 2) | 0.9158 | CatBoost |
| Fine-tuning Optuna (Fase 3) | 0.9161 | CatBoost / XGB |

### Principais Observações

- Os **três GBMs (CatBoost, XGB, LGBM)** tiveram desempenho muito próximo desde o baseline, com diferença menor que 0.002 entre eles.
- As estratégias de **feature engineering** não trouxeram ganho significativo em relação ao preprocessamento simples — indicando que os modelos já extraem bem a informação presente nos dados originais.

### Possíveis Melhorias

1. **Testar novas features** — as features criadas manualmente não agregaram valor. Vale explorar interações mais sofisticadas, como `MonthlyCharges × Contract`, ou features baseadas em clusters de comportamento de cliente.

2. **Aumentar os trials do Optuna** — com mais trials, o espaço de hiperparâmetros pode ser explorado com maior profundidade, com potencial real de ganho no AUC.

3. **Stacking (ensemble avançado)** — em vez de média simples, treinar um meta-modelo (ex: Regressão Logística) que aprende a combinar as previsões dos 3 GBMs. Historicamente traz +0.002 a +0.005 de AUC em relação ao voting simples.

4. **Pseudo-labeling** — usar o modelo treinado para gerar predições no test set, adicionar as predições de alta confiança ao treino e retreinar. Técnica comum nos tops de Kaggle em competições com dados sintéticos.